In [ ]:

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)




BASE_DIR = Path("data")
GDSC_DIR = BASE_DIR / "GDSC"
SCREEN_DIR = BASE_DIR / "celligner_mss_tmbh_pan_cancer_screen"
OUTPUT_DIR = SCREEN_DIR / "GDSC_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GDSC1_FILE = GDSC_DIR / "GDSC1_fitted_dose_response_27Oct23.xlsx"
GDSC2_FILE = GDSC_DIR / "GDSC2_fitted_dose_response_27Oct23.xlsx"
COMPOUND_FILE = GDSC_DIR / "screened_compounds_rel_8.5.csv"

CRC_PRED_FILE = SCREEN_DIR / "crc_all_cell_line_predictions.csv"
UCEC_PRED_FILE = SCREEN_DIR / "ucec_all_cell_line_predictions.csv"
COMBINED_PRED_FILE = SCREEN_DIR / "cell_line_subgroup_predictions.csv"


PRISM_RESULT_CANDIDATES = [
    SCREEN_DIR / "pan_cancer_tissue_adjusted_continuous_results.csv",
    SCREEN_DIR / "all_drug_continuous_probability_results.csv",
]




SCORE_COLUMN = "p_MSS_TMBH"
MIN_MODELS_PER_DRUG = 20
MIN_TISSUES_PER_DRUG = 3
RUN_SOURCE_TISSUE_EXCLUSION = True

print("GDSC directory:", GDSC_DIR)
print("Output directory:", OUTPUT_DIR)


In [ ]:

def load_predictions():
    parts = []

    if CRC_PRED_FILE.exists():
        x = pd.read_csv(CRC_PRED_FILE)
        if "classifier_source" not in x.columns:
            x["classifier_source"] = "CRC"
        parts.append(x)

    if UCEC_PRED_FILE.exists():
        x = pd.read_csv(UCEC_PRED_FILE)
        if "classifier_source" not in x.columns:
            x["classifier_source"] = "UCEC"
        parts.append(x)

    if parts:
        pred = pd.concat(parts, ignore_index=True)
    elif COMBINED_PRED_FILE.exists():
        pred = pd.read_csv(COMBINED_PRED_FILE)
    else:
        raise FileNotFoundError(
            "No prediction file found. Expected CRC/UCEC all-cell prediction files in "
            f"{SCREEN_DIR}"
        )

    if "DepMap_ID" not in pred.columns:
        for c in ["ModelID", "depmap_id", "DepMapID"]:
            if c in pred.columns:
                pred = pred.rename(columns={c: "DepMap_ID"})
                break

    required = {"DepMap_ID", "classifier_source", SCORE_COLUMN}
    missing = required - set(pred.columns)
    if missing:
        raise KeyError(f"Prediction table missing required columns: {sorted(missing)}")

    pred["classifier_source"] = (
        pred["classifier_source"].astype(str).str.upper().replace({"COADREAD": "CRC"})
    )
    pred["DepMap_ID"] = pred["DepMap_ID"].astype(str)

    pred = pred.drop_duplicates(["classifier_source", "DepMap_ID"]).reset_index(drop=True)
    return pred

predictions = load_predictions()
print("Predictions:", predictions.shape)
display(predictions.groupby("classifier_source")["DepMap_ID"].nunique().rename("n_models"))
display(predictions.head())


In [ ]:


TISSUE_CANDIDATES = [
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "PrimaryDisease",
    "lineage",
    "Lineage",
    "OncotreeCode",
]
TISSUE_COLUMN = next((c for c in TISSUE_CANDIDATES if c in predictions.columns), None)
print("Tissue column currently available in predictions:", TISSUE_COLUMN)


In [ ]:

gdsc1_raw = pd.read_excel(GDSC1_FILE)
gdsc2_raw = pd.read_excel(GDSC2_FILE)
compound_annotation = pd.read_csv(COMPOUND_FILE)

print("GDSC1 raw:", gdsc1_raw.shape)
print(gdsc1_raw.columns.tolist())
print("\nGDSC2 raw:", gdsc2_raw.shape)
print(gdsc2_raw.columns.tolist())
print("\nCompound annotation:", compound_annotation.shape)
print(compound_annotation.columns.tolist())

display(gdsc1_raw.head())


In [ ]:

def _norm_col(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def find_col(df, candidates, required=True):
    exact = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in exact:
            return exact[candidate.lower()]

    normalized = {_norm_col(c): c for c in df.columns}
    for candidate in candidates:
        if _norm_col(candidate) in normalized:
            return normalized[_norm_col(candidate)]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}. Available columns: {df.columns.tolist()}"
        )
    return None


def standardize_gdsc(df, dataset):
    cell_col = find_col(df, ["CELL_LINE_NAME", "cell_line_name", "model_name", "CELL_LINE"])
    cosmic_col = find_col(df, ["COSMIC_ID", "COSMICID", "cosmic_id"], required=False)
    sanger_col = find_col(df, ["SANGER_MODEL_ID", "model_id", "MODEL_ID"], required=False)
    drug_id_col = find_col(df, ["DRUG_ID", "drug_id", "compound_id"])
    drug_col = find_col(df, ["DRUG_NAME", "drug_name", "compound_name"])
    auc_col = find_col(df, ["AUC", "auc"])
    ln_ic50_col = find_col(df, ["LN_IC50", "ln_ic50", "IC50"], required=False)
    z_col = find_col(df, ["Z_SCORE", "z_score"], required=False)

    out = pd.DataFrame({
        "dataset": dataset,
        "gdsc_cell_line": df[cell_col].astype(str),
        "drug_id": df[drug_id_col],
        "drug": df[drug_col].astype(str),
        "auc": pd.to_numeric(df[auc_col], errors="coerce"),
    })

    if cosmic_col is not None:
        out["COSMIC_ID"] = pd.to_numeric(df[cosmic_col], errors="coerce").astype("Int64")
    else:
        out["COSMIC_ID"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

    if sanger_col is not None:
        out["SANGER_MODEL_ID"] = df[sanger_col].astype("string")
    else:
        out["SANGER_MODEL_ID"] = pd.Series(pd.NA, index=df.index, dtype="string")

    out["ln_ic50"] = (
        pd.to_numeric(df[ln_ic50_col], errors="coerce") if ln_ic50_col is not None else np.nan
    )
    out["z_score"] = (
        pd.to_numeric(df[z_col], errors="coerce") if z_col is not None else np.nan
    )

    out = out.dropna(subset=["auc", "drug_id", "drug", "gdsc_cell_line"])

    
    out = (
        out.groupby(
            ["dataset", "gdsc_cell_line", "COSMIC_ID", "SANGER_MODEL_ID", "drug_id", "drug"],
            dropna=False,
            as_index=False,
        )
        .agg(
            auc=("auc", "median"),
            ln_ic50=("ln_ic50", "median"),
            z_score=("z_score", "median"),
        )
    )
    return out


gdsc1 = standardize_gdsc(gdsc1_raw, "GDSC1")
gdsc2 = standardize_gdsc(gdsc2_raw, "GDSC2")
gdsc = pd.concat([gdsc1, gdsc2], ignore_index=True)

print("Standardized response table:", gdsc.shape)
display(
    gdsc.groupby("dataset").agg(
        n_models=("gdsc_cell_line", "nunique"),
        n_drugs=("drug_id", "nunique"),
        n_measurements=("auc", "size"),
    )
)


In [ ]:

def normalize_cell_name(x):
    if pd.isna(x):
        return pd.NA
    return re.sub(r"[^A-Z0-9]+", "", str(x).upper().strip())


def score_metadata_file(path):
    try:
        sample = pd.read_csv(path, nrows=3)
    except Exception:
        return -1
    cols = set(sample.columns)
    score = 0
    if any(c in cols for c in ["ModelID", "DepMap_ID", "DepMapID"]): score += 10
    if any(c in cols for c in ["StrippedCellLineName", "CellLineName", "CCLEName"]): score += 5
    if any(c in cols for c in ["COSMICID", "COSMIC_ID", "CosmicID"]): score += 4
    if any(c in cols for c in ["SangerModelID", "SANGER_MODEL_ID"]): score += 3
    if any(c in cols for c in TISSUE_CANDIDATES): score += 2
    return score


def auto_find_model_metadata():
    candidates = []
    seen = set()
    for root in [BASE_DIR, Path("combined_paper")]:
        if not root.exists():
            continue
        for pattern in ["**/Model.csv", "**/model.csv", "**/*Model*.csv", "**/*model_metadata*.csv"]:
            for path in root.glob(pattern):
                if path in seen or "GDSC_validation" in str(path):
                    continue
                seen.add(path)
                score = score_metadata_file(path)
                if score > 0:
                    candidates.append((score, path))
    candidates.sort(key=lambda x: (-x[0], len(str(x[1]))))
    if candidates:
        print("Candidate DepMap metadata files:")
        for score, path in candidates[:10]:
            print(f"  score={score:2d} {path}")
        return candidates[0][1]
    return None



DEPMAP_MODEL_FILE = None



if DEPMAP_MODEL_FILE is None:
    DEPMAP_MODEL_FILE = auto_find_model_metadata()

if DEPMAP_MODEL_FILE is None:
    raise FileNotFoundError(
        "Could not find DepMap model metadata automatically. Set DEPMAP_MODEL_FILE manually."
    )

print("\nUsing:", DEPMAP_MODEL_FILE)
models = pd.read_csv(DEPMAP_MODEL_FILE)
print(models.shape)
print(models.columns.tolist())


In [ ]:

def build_depmap_crosswalk(models, predictions):
    depmap_col = find_col(models, ["ModelID", "DepMap_ID", "DepMapID"])
    cosmic_col = find_col(models, ["COSMICID", "COSMIC_ID", "CosmicID"], required=False)
    sanger_col = find_col(models, ["SangerModelID", "SANGER_MODEL_ID"], required=False)
    tissue_col = next((c for c in TISSUE_CANDIDATES if c in models.columns), None)

    name_cols = [
        c for c in ["StrippedCellLineName", "CellLineName", "CCLEName", "ModelName"]
        if c in models.columns
    ]

    meta = models.copy()
    meta[depmap_col] = meta[depmap_col].astype(str)

    
    wanted = set(predictions["DepMap_ID"].astype(str))
    meta = meta[meta[depmap_col].isin(wanted)].copy()

    rows = []
    for _, row in meta.iterrows():
        depmap_id = str(row[depmap_col])
        cosmic = pd.NA
        if cosmic_col is not None:
            cosmic = pd.to_numeric(pd.Series([row[cosmic_col]]), errors="coerce").astype("Int64").iloc[0]

        sanger = pd.NA
        if sanger_col is not None and pd.notna(row[sanger_col]):
            sanger = str(row[sanger_col])

        tissue = row[tissue_col] if tissue_col is not None else pd.NA

        names = []
        for c in name_cols:
            if pd.notna(row[c]):
                value = str(row[c])
                names.append(normalize_cell_name(value))
                
                if c == "CCLEName" and "_" in value:
                    names.append(normalize_cell_name(value.split("_", 1)[0]))
        names = list(dict.fromkeys([n for n in names if pd.notna(n)])) or [pd.NA]

        for n in names:
            rows.append({
                "DepMap_ID": depmap_id,
                "depmap_name_norm": n,
                "COSMIC_ID": cosmic,
                "SANGER_MODEL_ID": sanger,
                "model_tissue": tissue,
            })

    return pd.DataFrame(rows).drop_duplicates()


depmap_xwalk = build_depmap_crosswalk(models, predictions)
print("Crosswalk models:", depmap_xwalk["DepMap_ID"].nunique())
display(depmap_xwalk.head())


In [ ]:

def unique_lookup(df, key):
    x = df.dropna(subset=[key, "DepMap_ID"])[[key, "DepMap_ID"]].drop_duplicates()
    counts = x.groupby(key)["DepMap_ID"].nunique()
    valid = counts[counts == 1].index
    return x[x[key].isin(valid)].drop_duplicates(key).set_index(key)["DepMap_ID"].to_dict()

cosmic_lookup = unique_lookup(depmap_xwalk, "COSMIC_ID")
sanger_lookup = unique_lookup(depmap_xwalk, "SANGER_MODEL_ID")
name_lookup = unique_lookup(depmap_xwalk, "depmap_name_norm")


def map_gdsc_to_depmap(df):
    x = df.copy()
    x["gdsc_name_norm"] = x["gdsc_cell_line"].map(normalize_cell_name)
    x["DepMap_ID"] = pd.NA
    x["mapping_method"] = pd.NA

    mapped = x["COSMIC_ID"].map(cosmic_lookup)
    mask = x["DepMap_ID"].isna() & mapped.notna()
    x.loc[mask, "DepMap_ID"] = mapped[mask]
    x.loc[mask, "mapping_method"] = "COSMIC_ID"

    mapped = x["SANGER_MODEL_ID"].map(sanger_lookup)
    mask = x["DepMap_ID"].isna() & mapped.notna()
    x.loc[mask, "DepMap_ID"] = mapped[mask]
    x.loc[mask, "mapping_method"] = "SANGER_MODEL_ID"

    mapped = x["gdsc_name_norm"].map(name_lookup)
    mask = x["DepMap_ID"].isna() & mapped.notna()
    x.loc[mask, "DepMap_ID"] = mapped[mask]
    x.loc[mask, "mapping_method"] = "cell_line_name"

    return x


gdsc_mapped = map_gdsc_to_depmap(gdsc)

model_level_mapping = gdsc_mapped[
    ["dataset", "gdsc_cell_line", "DepMap_ID", "mapping_method"]
].drop_duplicates()

mapping_summary = model_level_mapping.groupby("dataset").agg(
    n_gdsc_models=("gdsc_cell_line", "nunique"),
    n_mapped_depmap_models=("DepMap_ID", "nunique"),
)
mapping_summary["mapped_fraction"] = (
    mapping_summary["n_mapped_depmap_models"] / mapping_summary["n_gdsc_models"]
)

display(mapping_summary)
display(model_level_mapping["mapping_method"].value_counts(dropna=False).rename("n_models"))

unmapped = gdsc_mapped.loc[
    gdsc_mapped["DepMap_ID"].isna(),
    ["dataset", "gdsc_cell_line", "COSMIC_ID", "SANGER_MODEL_ID"],
].drop_duplicates().sort_values(["dataset", "gdsc_cell_line"])

print("Unmapped GDSC models:", len(unmapped))
display(unmapped.head(50))
unmapped.to_csv(OUTPUT_DIR / "unmapped_gdsc_models.csv", index=False)


In [ ]:


if (mapping_summary["mapped_fraction"] < 0.40).any():
    print(
        "WARNING: <40% of GDSC models mapped in at least one dataset. "
        "Inspect the metadata mapping before interpreting results."
    )


model_tissue = depmap_xwalk[["DepMap_ID", "model_tissue"]].dropna().drop_duplicates()
tissue_counts = model_tissue.groupby("DepMap_ID")["model_tissue"].nunique()
model_tissue = model_tissue[
    model_tissue["DepMap_ID"].isin(tissue_counts[tissue_counts == 1].index)
].drop_duplicates("DepMap_ID")

if TISSUE_COLUMN is None:
    predictions = predictions.merge(
        model_tissue, on="DepMap_ID", how="left", validate="many_to_one"
    )
    TISSUE_COLUMN = "model_tissue"

print("Tissue column used:", TISSUE_COLUMN)
print("Missing tissue fraction:", predictions[TISSUE_COLUMN].isna().mean())


In [ ]:

gdsc_mapped = gdsc_mapped.dropna(subset=["DepMap_ID"]).copy()
gdsc_mapped["DepMap_ID"] = gdsc_mapped["DepMap_ID"].astype(str)

score_cols = ["DepMap_ID", "classifier_source", SCORE_COLUMN, TISSUE_COLUMN]
for c in ["p_MSS_TMBL", "p_MSIH", "probability_margin", "predicted_group"]:
    if c in predictions.columns:
        score_cols.append(c)

score_table = predictions[score_cols].drop_duplicates(["classifier_source", "DepMap_ID"])

parts = []
for source, score_df in score_table.groupby("classifier_source", sort=False):
    x = gdsc_mapped.merge(score_df, on="DepMap_ID", how="inner", validate="many_to_one")
    parts.append(x)

gdsc_scores = pd.concat(parts, ignore_index=True)

coverage = gdsc_scores.groupby(["dataset", "classifier_source"]).agg(
    n_models=("DepMap_ID", "nunique"),
    n_drugs=("drug_id", "nunique"),
    n_tissues=(TISSUE_COLUMN, "nunique"),
    n_measurements=("auc", "size"),
).reset_index()

display(coverage)
coverage.to_csv(OUTPUT_DIR / "gdsc_coverage_summary.csv", index=False)
gdsc_scores.to_csv(OUTPUT_DIR / "gdsc_model_score_merged.csv", index=False)


In [ ]:

def source_tissue_mask(df, classifier_source):
    tissue = df[TISSUE_COLUMN].astype(str).str.lower()
    if classifier_source == "CRC":
        return tissue.str.contains(
            r"bowel|colon|rect|colorectal|large intestine", regex=True, na=False
        )
    if classifier_source == "UCEC":
        return tissue.str.contains(
            r"uter|endometr", regex=True, na=False
        )
    return pd.Series(False, index=df.index)


def run_one_screen(data, analysis_set, score=SCORE_COLUMN):
    rows = []

    for (dataset, source, drug_id, drug), drug_df in data.groupby(
        ["dataset", "classifier_source", "drug_id", "drug"], sort=False
    ):
        x = drug_df[["DepMap_ID", TISSUE_COLUMN, score, "auc"]].dropna().copy()
        x = x.drop_duplicates("DepMap_ID")

        if analysis_set == "exclude_source_tissue":
            x = x.loc[~source_tissue_mask(x, source)].copy()

        
        tissue_n = x.groupby(TISSUE_COLUMN)["DepMap_ID"].nunique()
        eligible_tissues = tissue_n[tissue_n >= 2].index
        x = x[x[TISSUE_COLUMN].isin(eligible_tissues)].copy()

        n_models = x["DepMap_ID"].nunique()
        n_tissues = x[TISSUE_COLUMN].nunique()
        if n_models < MIN_MODELS_PER_DRUG or n_tissues < MIN_TISSUES_PER_DRUG:
            continue

        x["score_within_tissue"] = x[score] - x.groupby(TISSUE_COLUMN)[score].transform("mean")
        x["auc_within_tissue"] = x["auc"] - x.groupby(TISSUE_COLUMN)["auc"].transform("mean")

        if np.isclose(x["score_within_tissue"].abs().sum(), 0):
            continue

        fit = sm.OLS(
            x["auc_within_tissue"].astype(float),
            x[["score_within_tissue"]].astype(float),
        ).fit(cov_type="HC3")

        rho, rho_p = spearmanr(
            x["score_within_tissue"], x["auc_within_tissue"], nan_policy="omit"
        )

        rows.append({
            "analysis_set": analysis_set,
            "dataset": dataset,
            "classifier_source": source,
            "score": score,
            "drug_id": drug_id,
            "drug": drug,
            "n_models": n_models,
            "n_tissues": n_tissues,
            "beta_tissue_adjusted": fit.params["score_within_tissue"],
            "standard_error_HC3": fit.bse["score_within_tissue"],
            "pvalue": fit.pvalues["score_within_tissue"],
            "spearman_rho_within_tissue": rho,
            "spearman_pvalue_within_tissue": rho_p,
            "median_auc": x["auc"].median(),
        })

    return pd.DataFrame(rows)


result_parts = [run_one_screen(gdsc_scores, "all_lineages")]
if RUN_SOURCE_TISSUE_EXCLUSION:
    result_parts.append(run_one_screen(gdsc_scores, "exclude_source_tissue"))

gdsc_results = pd.concat(result_parts, ignore_index=True)

if gdsc_results.empty:
    raise RuntimeError("No GDSC tests passed coverage requirements.")


gdsc_results["FDR_within_analysis"] = np.nan
for _, idx in gdsc_results.groupby(
    ["analysis_set", "dataset", "classifier_source"]
).groups.items():
    idx = list(idx)
    gdsc_results.loc[idx, "FDR_within_analysis"] = multipletests(
        gdsc_results.loc[idx, "pvalue"], method="fdr_bh"
    )[1]


gdsc_results["FDR_global_GDSC"] = multipletests(
    gdsc_results["pvalue"], method="fdr_bh"
)[1]

gdsc_results["direction"] = np.where(
    gdsc_results["beta_tissue_adjusted"] < 0, "sensitivity", "resistance"
)

gdsc_results = gdsc_results.sort_values(["FDR_within_analysis", "pvalue"]).reset_index(drop=True)

display(
    gdsc_results.groupby(["analysis_set", "dataset", "classifier_source"]).agg(
        n_tests=("drug_id", "size"),
        n_FDR05=("FDR_within_analysis", lambda x: int((x < 0.05).sum())),
        min_FDR=("FDR_within_analysis", "min"),
    )
)

display(gdsc_results.head(50))
gdsc_results.to_csv(OUTPUT_DIR / "gdsc_continuous_tissue_adjusted_results.csv", index=False)


In [ ]:

sig = gdsc_results[gdsc_results["FDR_within_analysis"] < 0.05].copy()

cols = [
    "analysis_set", "dataset", "classifier_source", "drug",
    "n_models", "n_tissues", "beta_tissue_adjusted",
    "pvalue", "FDR_within_analysis", "FDR_global_GDSC", "direction",
]
display(sig[cols].head(100))
sig.to_csv(OUTPUT_DIR / "gdsc_significant_results.csv", index=False)


In [ ]:

def normalize_drug_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower().strip())

primary = gdsc_results[gdsc_results["analysis_set"] == "all_lineages"].copy()
primary["drug_key"] = primary["drug"].map(normalize_drug_name)

replication_parts = []
for source, x in primary.groupby("classifier_source", sort=False):
    g1 = x[x["dataset"] == "GDSC1"].drop_duplicates("drug_key")
    g2 = x[x["dataset"] == "GDSC2"].drop_duplicates("drug_key")

    m = g1.merge(g2, on="drug_key", how="inner", suffixes=("_GDSC1", "_GDSC2"))
    if m.empty:
        continue

    out = pd.DataFrame({
        "classifier_source": source,
        "drug_key": m["drug_key"],
        "drug": m["drug_GDSC2"],
        "beta_GDSC1": m["beta_tissue_adjusted_GDSC1"],
        "beta_GDSC2": m["beta_tissue_adjusted_GDSC2"],
        "pvalue_GDSC1": m["pvalue_GDSC1"],
        "pvalue_GDSC2": m["pvalue_GDSC2"],
        "FDR_GDSC1": m["FDR_within_analysis_GDSC1"],
        "FDR_GDSC2": m["FDR_within_analysis_GDSC2"],
        "n_models_GDSC1": m["n_models_GDSC1"],
        "n_models_GDSC2": m["n_models_GDSC2"],
    })
    out["same_direction"] = np.sign(out["beta_GDSC1"]) == np.sign(out["beta_GDSC2"])
    out["both_more_sensitive"] = (out["beta_GDSC1"] < 0) & (out["beta_GDSC2"] < 0)
    out["both_more_resistant"] = (out["beta_GDSC1"] > 0) & (out["beta_GDSC2"] > 0)
    out["significant_GDSC1"] = out["FDR_GDSC1"] < 0.05
    out["significant_GDSC2"] = out["FDR_GDSC2"] < 0.05
    out["significant_both"] = out["significant_GDSC1"] & out["significant_GDSC2"]
    replication_parts.append(out)

if replication_parts:
    gdsc_replication = pd.concat(replication_parts, ignore_index=True)
    gdsc_replication = gdsc_replication.sort_values(
        ["significant_both", "same_direction", "both_more_sensitive"],
        ascending=[False, False, False],
    )
    display(
        gdsc_replication.groupby("classifier_source").agg(
            n_common_drugs=("drug_key", "size"),
            n_same_direction=("same_direction", "sum"),
            n_significant_both=("significant_both", "sum"),
        )
    )
    display(gdsc_replication.head(100))
else:
    gdsc_replication = pd.DataFrame()

gdsc_replication.to_csv(OUTPUT_DIR / "gdsc1_gdsc2_replication.csv", index=False)


In [ ]:

PREVIOUS_CANDIDATES = [
    "beta-lapachone", "tasisulam", "sangivamycin", "lacitol", "LY2801653",
    "pilaralisib", "indisulam", "M-344", "oligomycin A",
    "dofetilide", "FG-4592", "SB-228357",
    "taselisib", "buparlisib", "palbociclib", "ribociclib", "abemaciclib",
    "MK-1775", "LY2603618", "sulfasalazine", "erastin",
]

candidate_lookup = {normalize_drug_name(x): x for x in PREVIOUS_CANDIDATES}
all_gdsc_drug_keys = set(gdsc["drug"].map(normalize_drug_name))

availability = pd.DataFrame({"candidate": PREVIOUS_CANDIDATES})
availability["available_in_GDSC"] = availability["candidate"].map(normalize_drug_name).isin(all_gdsc_drug_keys)
display(availability)
availability.to_csv(OUTPUT_DIR / "previous_candidate_gdsc_availability.csv", index=False)

candidate_results = gdsc_results.copy()
candidate_results["drug_key"] = candidate_results["drug"].map(normalize_drug_name)
candidate_results = candidate_results[candidate_results["drug_key"].isin(candidate_lookup)].copy()
candidate_results["candidate_requested_name"] = candidate_results["drug_key"].map(candidate_lookup)

candidate_view = candidate_results.sort_values(
    ["candidate_requested_name", "analysis_set", "classifier_source", "dataset"]
)
display(candidate_view[
    [
        "candidate_requested_name", "drug", "analysis_set", "dataset",
        "classifier_source", "n_models", "n_tissues", "beta_tissue_adjusted",
        "pvalue", "FDR_within_analysis", "direction",
    ]
])
candidate_view.to_csv(OUTPUT_DIR / "previous_candidate_drug_validation.csv", index=False)


In [ ]:

PRISM_FILE = next((p for p in PRISM_RESULT_CANDIDATES if p.exists()), None)
print("PRISM file:", PRISM_FILE)

prism_gdsc = pd.DataFrame()
if PRISM_FILE is not None:
    prism = pd.read_csv(PRISM_FILE)

    if "analysis_set" in prism.columns:
        prism = prism[prism["analysis_set"] == "all_lineages"].copy()
    if "score" in prism.columns:
        prism = prism[prism["score"] == SCORE_COLUMN].copy()
    if "drug" not in prism.columns and "drug_name" in prism.columns:
        prism = prism.rename(columns={"drug_name": "drug"})

    required = {"classifier_source", "drug", "beta_tissue_adjusted", "pvalue"}
    missing = required - set(prism.columns)

    if missing:
        print("Skipping PRISM comparison; missing columns:", sorted(missing))
    else:
        prism["classifier_source"] = (
            prism["classifier_source"].astype(str).str.upper().replace({"COADREAD": "CRC"})
        )
        prism["drug_key"] = prism["drug"].map(normalize_drug_name)

        fdr_col = next(
            (c for c in ["FDR_within_analysis", "FDR_global", "FDR_global_all_tests"] if c in prism.columns),
            None,
        )
        keep = ["classifier_source", "drug_key", "drug", "beta_tissue_adjusted", "pvalue"]
        if fdr_col is not None:
            keep.append(fdr_col)

        prism_small = prism[keep].drop_duplicates(["classifier_source", "drug_key"]).copy()
        rename_map = {
            "drug": "drug_PRISM",
            "beta_tissue_adjusted": "beta_PRISM",
            "pvalue": "pvalue_PRISM",
        }
        if fdr_col is not None:
            rename_map[fdr_col] = "FDR_PRISM"
        prism_small = prism_small.rename(columns=rename_map)

        g = primary.copy()
        g["drug_key"] = g["drug"].map(normalize_drug_name)
        prism_gdsc = g.merge(prism_small, on=["classifier_source", "drug_key"], how="inner")
        prism_gdsc["same_direction_PRISM_GDSC"] = (
            np.sign(prism_gdsc["beta_tissue_adjusted"]) == np.sign(prism_gdsc["beta_PRISM"])
        )
        prism_gdsc["both_sensitivity_PRISM_GDSC"] = (
            (prism_gdsc["beta_tissue_adjusted"] < 0) & (prism_gdsc["beta_PRISM"] < 0)
        )

        display(
            prism_gdsc.groupby(["dataset", "classifier_source"]).agg(
                n_common_drugs=("drug_key", "size"),
                n_same_direction=("same_direction_PRISM_GDSC", "sum"),
                n_both_sensitivity=("both_sensitivity_PRISM_GDSC", "sum"),
            )
        )
        display(
            prism_gdsc.sort_values(
                ["same_direction_PRISM_GDSC", "FDR_within_analysis"],
                ascending=[False, True],
            ).head(100)
        )
        prism_gdsc.to_csv(OUTPUT_DIR / "prism_gdsc_consistency.csv", index=False)
else:
    print("No previous PRISM continuous result file found; this section is skipped.")


In [ ]:


consensus_parts = []
for source, x in primary.groupby("classifier_source", sort=False):
    xx = x.copy()
    xx["drug_key"] = xx["drug"].map(normalize_drug_name)

    beta = xx.pivot_table(
        index="drug_key", columns="dataset", values="beta_tissue_adjusted", aggfunc="first"
    ).rename(columns={"GDSC1": "beta_GDSC1", "GDSC2": "beta_GDSC2"})
    fdr = xx.pivot_table(
        index="drug_key", columns="dataset", values="FDR_within_analysis", aggfunc="first"
    ).rename(columns={"GDSC1": "FDR_GDSC1", "GDSC2": "FDR_GDSC2"})
    names = xx.groupby("drug_key")["drug"].first().rename("drug")

    one = pd.concat([names, beta, fdr], axis=1).reset_index()
    one["classifier_source"] = source
    consensus_parts.append(one)

consensus = pd.concat(consensus_parts, ignore_index=True)

if not prism_gdsc.empty:
    prism_cols = ["classifier_source", "drug_key", "beta_PRISM"]
    if "FDR_PRISM" in prism_gdsc.columns:
        prism_cols.append("FDR_PRISM")
    prism_for_consensus = prism_gdsc[prism_cols].drop_duplicates(["classifier_source", "drug_key"])
    consensus = consensus.merge(
        prism_for_consensus, on=["classifier_source", "drug_key"], how="left"
    )

beta_cols = [c for c in ["beta_PRISM", "beta_GDSC1", "beta_GDSC2"] if c in consensus.columns]


def summarize_directions(row):
    vals = [row[c] for c in beta_cols if pd.notna(row[c])]
    if not vals:
        return pd.Series({
            "n_platforms_available": 0,
            "n_sensitivity_direction": 0,
            "n_resistance_direction": 0,
            "all_same_direction": False,
        })
    signs = np.sign(vals)
    return pd.Series({
        "n_platforms_available": len(vals),
        "n_sensitivity_direction": int(sum(v < 0 for v in vals)),
        "n_resistance_direction": int(sum(v > 0 for v in vals)),
        "all_same_direction": len(set(signs)) == 1,
    })

consensus = pd.concat([consensus, consensus.apply(summarize_directions, axis=1)], axis=1)
consensus = consensus.sort_values(
    ["n_sensitivity_direction", "all_same_direction", "n_platforms_available"],
    ascending=[False, False, False],
)

display(consensus.head(100))
consensus.to_csv(OUTPUT_DIR / "prism_gdsc_consensus.csv", index=False)


In [ ]:


ann = compound_annotation.copy()
ann_drug_col = find_col(ann, ["drug_name", "DRUG_NAME", "compound_name", "name"], required=False)

if ann_drug_col is not None:
    ann["drug_key"] = ann[ann_drug_col].map(normalize_drug_name)
    ann = ann.drop_duplicates("drug_key")

    useful = [
        c for c in ann.columns
        if any(token in c.lower() for token in ["target", "pathway", "synonym", "drug_name", "compound_name"])
    ]
    useful = list(dict.fromkeys(["drug_key"] + useful))

    gdsc_results_annotated = gdsc_results.copy()
    gdsc_results_annotated["drug_key"] = gdsc_results_annotated["drug"].map(normalize_drug_name)
    gdsc_results_annotated = gdsc_results_annotated.merge(
        ann[useful], on="drug_key", how="left"
    )
    gdsc_results_annotated.to_csv(
        OUTPUT_DIR / "gdsc_continuous_tissue_adjusted_results_annotated.csv", index=False
    )
    display(gdsc_results_annotated.head(30))
else:
    print("Could not identify a drug-name field in compound annotation; skipping annotation merge.")


In [ ]:

print("Primary files written to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", p.name)

print("\nInterpretation reminder:")
print("  beta < 0  -> higher MSS/TMB-H similarity, lower AUC, greater sensitivity")
print("  beta > 0  -> higher MSS/TMB-H similarity, higher AUC, relative resistance")
